# 사이클 단위 분석 — 탱크별 온도 vs 제어요소

`Now_Actual_Wagon_Num`(1→42 증가 후 1로 리셋)을 기준으로 한 사이클씩 끊어서 봅니다.

**사용법**
1. 아래 `0. 설정` ~ `1. 전체 로드` 셀을 순서대로 한 번 실행
2. 각 탱크 섹션에서 `cycle_table(...)` 로 사이클 목록을 보고, 보고 싶은 번호를 골라
   `CYCLE` 변수에 넣고 `plot_cycle(...)` 셀을 다시 실행
3. 여러 사이클을 겹쳐 비교하려면 맨 아래 `overlay_cycles(...)` 사용

데이터는 `Tank_Temp_Factor_Analysis.py` 가 저장한 `extracted_csv/*_Temp_vs_Factors_*.csv`
중 **가장 최신 파일**을 자동으로 씁니다. 새로 받으셨으면 `load_tank(탱크, refresh=True)`.

⚠️ 이 노트북은 **parkkt 폴더에서 실행**된다고 가정합니다.

## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
# 탱크별 태그 매핑은 추출기 쪽 한 곳에서만 관리한다 (중복 정의 금지)
from Tank_Temp_Factor_Analysis import TANKS, FACTOR_NAMES, WAGON_TAG

import matplotlib
import matplotlib.pyplot as plt
# ↑ 위 import 가 백엔드를 Agg 로 바꿔놓기 때문에, inline 은 반드시 그 **뒤**에 와야 한다
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "extracted_csv"
COLORS = {"heat": "#d62728", "cool": "#1f77b4", "exchanger": "#2ca02c", "feeding": "#9467bd"}

In [ ]:
_CACHE = {}


def load_tank(tank_key, refresh=False):
    """탱크 CSV 를 읽고 wagon 카운터로 사이클 번호를 붙인다.

    wagon 값이 **줄어드는 순간**(42 -> 1)이 사이클 경계다. 라인이 멈췄다 다시
    돌면 중간에 몇 분씩 비는 사이클도 생기므로, 길이는 사이클마다 제각각이다.
    """
    if refresh or tank_key not in _CACHE:
        path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")))[-1]
        df = pd.read_csv(path, index_col="Time", parse_dates=True).sort_index()
        df["cycle"] = (df[WAGON_TAG].diff() < 0).cumsum()
        _CACHE[tank_key] = df
        print(f"[{tank_key}] {os.path.basename(path)} — {df.index.min()} ~ {df.index.max()} "
              f"| {len(df):,}행 | 사이클 {df['cycle'].nunique()}개")
    return _CACHE[tank_key]

In [ ]:
def cycle_table(tank_key):
    """사이클 목록 — 언제 시작해서 몇 분 걸렸고, 온도가 어디서 어디로 갔는지."""
    df = load_tank(tank_key)
    cfg = TANKS[tank_key]
    g = df.groupby("cycle")

    t = pd.DataFrame({
        "시작": g.apply(lambda x: x.index[0]),
        "끝": g.apply(lambda x: x.index[-1]),
        "wagon": g[WAGON_TAG].min().astype(int).astype(str) + "~" + g[WAGON_TAG].max().astype(int).astype(str),
        "행수": g.size(),
        "온도시작": g[cfg["temp"]].first().round(0),
        "온도끝": g[cfg["temp"]].last().round(0),
        "온도최저": g[cfg["temp"]].min().round(0),
        "온도최고": g[cfg["temp"]].max().round(0),
    })
    t["분"] = ((t["끝"] - t["시작"]).dt.total_seconds() / 60).round(1)
    for n in FACTOR_NAMES:
        t[f"{n}_ON%"] = (g[cfg[n]].mean() * 100).round(1)

    return t[["시작", "끝", "분", "wagon", "행수", "온도시작", "온도끝", "온도최저", "온도최고"]
             + [f"{n}_ON%" for n in FACTOR_NAMES]]

In [ ]:
def plot_cycle(tank_key, cycle_no):
    """사이클 하나를 온도 + 4개 제어요소 + wagon 번호로 나눠서 그린다."""
    df = load_tank(tank_key)
    cfg = TANKS[tank_key]
    d = df[df["cycle"] == cycle_no]
    if d.empty:
        print(f"⚠️ [{tank_key}] 사이클 {cycle_no} 없음")
        return

    n = len(FACTOR_NAMES)
    fig, axes = plt.subplots(n + 2, 1, sharex=True, figsize=(15, 3 + 1.2 * (n + 1)),
                             gridspec_kw={"height_ratios": [3] + [1] * n + [1.2]})

    ax = axes[0]
    ax.plot(d.index, d[cfg["temp"]], color="#d62728", linewidth=1.2)
    ax.set_ylabel(cfg["temp"], fontsize=9)
    mins = (d.index[-1] - d.index[0]).total_seconds() / 60
    ax.set_title(f"{tank_key} — 사이클 {cycle_no}   {d.index[0]} ~ {d.index[-1]}   ({mins:.1f}분, {len(d):,}행)")
    ax.grid(alpha=0.3)

    for a, name in zip(axes[1:1 + n], FACTOR_NAMES):
        a.step(d.index, d[cfg[name]], where="post", color=COLORS[name], linewidth=1.2)
        a.set_ylim(-0.2, 1.2)
        a.set_yticks([0, 1])
        a.set_ylabel(name, fontsize=9)
        a.grid(alpha=0.3)

    aw = axes[-1]
    aw.step(d.index, d[WAGON_TAG], where="post", color="#555555", linewidth=1.2)
    aw.set_ylabel("wagon", fontsize=9)
    aw.grid(alpha=0.3)

    fig.autofmt_xdate()
    fig.tight_layout()

In [ ]:
def overlay_cycles(tank_key, cycle_nos, factor=None):
    """여러 사이클을 **wagon 번호 기준으로 정렬**해 겹쳐 그린다.

    시간축으로 보면 사이클마다 길이가 달라 비교가 안 되는데, wagon 번호로 맞추면
    "공정상 같은 지점"끼리 비교된다. factor 를 주면 그 요소의 ON 비율도 같이 본다.
    """
    df = load_tank(tank_key)
    cfg = TANKS[tank_key]
    rows = 2 if factor else 1
    fig, axes = plt.subplots(rows, 1, sharex=True, figsize=(13, 4 + 2 * (rows - 1)), squeeze=False)

    for c in cycle_nos:
        d = df[df["cycle"] == c]
        if d.empty:
            continue
        prof = d.groupby(d[WAGON_TAG])[cfg["temp"]].mean()
        axes[0][0].plot(prof.index, prof.values, linewidth=1.2, alpha=0.8, label=f"사이클 {c}")
        if factor:
            fp = d.groupby(d[WAGON_TAG])[cfg[factor]].mean()
            axes[1][0].plot(fp.index, fp.values, linewidth=1.2, alpha=0.8)

    axes[0][0].set_ylabel(cfg["temp"], fontsize=9)
    axes[0][0].set_title(f"{tank_key} — 사이클 비교 (wagon 번호 기준 정렬)")
    axes[0][0].legend(fontsize=8, ncol=4)
    axes[0][0].grid(alpha=0.3)
    if factor:
        axes[1][0].set_ylabel(f"{factor} ON 비율", fontsize=9)
        axes[1][0].grid(alpha=0.3)
    axes[-1][0].set_xlabel("wagon 번호 (1~42)")
    fig.tight_layout()

## 1. 전체 탱크 로드

In [ ]:
for tk in TANKS:
    load_tank(tk)

## 2. P1 — 워킹 Tank 1 BACK

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("P1").tail(15)

In [ ]:
P1_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("P1", P1_CYCLE)

## 3. P2 — 워킹 Tank 2-1 SOFT

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("P2").tail(15)

In [ ]:
P2_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("P2", P2_CYCLE)

## 4. P3 — 워킹 Tank 3-1 고탄성

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("P3").tail(15)

In [ ]:
P3_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("P3", P3_CYCLE)

## 5. P4 — 워킹 Tank 2-2 HARD

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("P4").tail(15)

In [ ]:
P4_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("P4", P4_CYCLE)

## 6. P5 — 워킹 Tank 3-2 CUSH

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("P5").tail(15)

In [ ]:
P5_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("P5", P5_CYCLE)

## 7. I1 — 워킹 Tank 1 ISO

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("I1").tail(15)

In [ ]:
I1_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("I1", I1_CYCLE)

## 8. I2 — 워킹 Tank 2 MDI

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("I2").tail(15)

In [ ]:
I2_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("I2", I2_CYCLE)

## 9. I3 — 워킹 Tank 3 ISO

In [ ]:
# 사이클 목록 (뒤쪽 15개). 전체를 보려면 .tail(15) 를 떼세요
cycle_table("I3").tail(15)

In [ ]:
I3_CYCLE = 1   # 위 표에서 보고 싶은 사이클 번호로 바꾸세요

plot_cycle("I3", I3_CYCLE)

## 10. 사이클 겹쳐 보기 (선택)

시간축 대신 **wagon 번호**로 맞춰서 여러 사이클을 겹쳐 봅니다.
사이클마다 길이가 달라도 "공정상 같은 지점"끼리 비교됩니다.

In [ ]:
OVERLAY_TANK = "P1"
OVERLAY_CYCLES = [1, 2, 3, 4, 5]   # 비교할 사이클 번호들
OVERLAY_FACTOR = "heat"            # None 이면 온도만

overlay_cycles(OVERLAY_TANK, OVERLAY_CYCLES, factor=OVERLAY_FACTOR)